# Nền tảng 5 — Unicode và chính tả tiếng Việt

Một nửa project là về **NFD**: điều gì xảy ra nếu tách dấu ra khỏi chữ trước khi huấn luyện tokenizer. Muốn phát
biểu bất cứ điều gì về nửa đó, phải nắm chắc tầng dưới: code point, byte, chuẩn hoá Unicode, và cách tiếng Việt
viết thanh điệu.

Notebook này đi từ byte lên tới giả thuyết H2 và H4, và chỉ ra ba chỗ dễ sai:

- đếm "ký tự" trên dạng NFD ra kết quả khác NFC, làm hỏng mẫu số của bpc;
- so hai chuỗi trông giống hệt nhau bằng `==` ra `False`;
- coi `đ` là "d có dấu", trong khi nó là một chữ cái riêng.

**Chạy bằng kernel pixi của project.**

In [ ]:
import json
import math
import sys
import unicodedata
from collections import Counter
from pathlib import Path

import regex as re
from tokenizers import Tokenizer

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "kaggle" / "outputs" / "vitok-data"

from vitok.text import TONE_MARKS, nfc, nfd, strip_diacritics, strip_diacritics_partial, tone_of, with_tone

docs = [json.loads(l)["text"] for l in (DATA / "test.jsonl").open(encoding="utf-8")]
print(f"{len(docs):,} văn bản test đã nạp")

## 1. Ba mức: code point, ký tự, byte

Ba khái niệm hay bị gọi lẫn là "ký tự":

- **Code point**: một số nguyên Unicode gán cho một ký hiệu, viết là `U+1EC7`. Python đếm `len(s)` theo code point.
- **Ký tự cảm nhận được** (grapheme cluster): cái người đọc coi là một chữ. Có thể gồm nhiều code point.
- **Byte**: biểu diễn thực tế trên đĩa, phụ thuộc bảng mã. Project dùng UTF-8.

UTF-8 mã hoá một code point thành 1–4 byte theo quy tắc:

| Khoảng code point | Số byte | Dạng bit |
|---|---|---|
| U+0000–U+007F | 1 | `0xxxxxxx` |
| U+0080–U+07FF | 2 | `110xxxxx 10xxxxxx` |
| U+0800–U+FFFF | 3 | `1110xxxx 10xxxxxx 10xxxxxx` |
| U+10000–U+10FFFF | 4 | `11110xxx 10xxxxxx 10xxxxxx 10xxxxxx` |

Byte tiếp theo luôn bắt đầu bằng `10`, nên nhìn một byte bất kỳ là biết nó mở đầu hay đi tiếp — tính chất làm
UTF-8 tự đồng bộ được.

Ở dạng NFC, chữ Việt có dấu rơi vào **ba** khối Unicode khác nhau, nên số byte không đồng đều:

| Khối | Ví dụ | Byte UTF-8 |
|---|---|---|
| Latin-1 Supplement (U+00C0–U+00FF) | `à á â ã è é ê ì í ò ó ô õ ù ú ý` | 2 |
| Latin Extended-A (U+0100–U+017F) | `ă đ ĩ ũ` | 2 |
| Latin Extended-B (U+0180–U+024F) | `ơ ư` | 2 |
| Latin Extended Additional (U+1EA0–U+1EF9) | `ạ ả ấ ầ ẩ ẫ ậ ệ ố ở ự ...` | 3 |

Chữ không dấu và dấu cách là ASCII, 1 byte. Vì phần lớn ký tự trong văn bản là ASCII, trung bình văn bản tiếng
Việt chỉ tốn khoảng 1,3 byte mỗi ký tự. Cell dưới đếm phân bố đó trên dữ liệu thật.

In [ ]:
for ch in "aáơệđ":
    b = ch.encode("utf-8")
    print(f"  {ch!r}  U+{ord(ch):04X}  {unicodedata.name(ch):<45s} {len(b)} byte  {' '.join(f'{x:08b}' for x in b)}")

mau = "".join(docs[:200])
mau_nfc, mau_nfd = nfc(mau), nfd(mau)

phan_bo = Counter(len(c.encode()) for c in mau_nfc)
print("\nphân bố số byte UTF-8 của từng ký tự NFC (200 văn bản test):")
for k in sorted(phan_bo):
    print(f"  {k} byte: {phan_bo[k] / len(mau_nfc):6.1%} số ký tự")
print(f"\ntrên 200 văn bản test:")
print(f"  code point NFC : {len(mau_nfc):,}")
print(f"  code point NFD : {len(mau_nfd):,}  ({len(mau_nfd) / len(mau_nfc) - 1:+.1%})")
print(f"  byte UTF-8 NFC : {len(mau_nfc.encode()):,}  ({len(mau_nfc.encode()) / len(mau_nfc):.3f} byte mỗi ký tự)")
print(f"  byte UTF-8 NFD : {len(mau_nfd.encode()):,}  ({len(mau_nfd.encode()) / len(mau_nfc.encode()) - 1:+.1%})")

## 2. Chuẩn hoá: cùng một chữ, hai cách mã hoá

Unicode cho phép viết "ệ" theo hai cách:

1. **Dựng sẵn** (precomposed): một code point duy nhất `U+1EC7`.
2. **Tổ hợp** (decomposed): `e` + `U+0302` (dấu mũ) + `U+0323` (dấu nặng).

Hai dãy này **tương đương chính tắc** (canonically equivalent): theo chuẩn Unicode chúng là *cùng một văn bản*.
Nhưng chúng là hai dãy số khác nhau, nên `==` trong Python trả về `False`, `len` khác nhau, và một tokenizer huấn
luyện trên dạng này sẽ không nhận ra dạng kia.

Bốn dạng chuẩn hoá:

| Dạng | Làm gì | Dùng khi nào |
|---|---|---|
| **NFD** | tách hết ra dạng tổ hợp | khi muốn thao tác trên dấu — nhánh NFD của project |
| **NFC** | tách ra rồi dựng lại tối đa | mặc định của web và của mọi thứ khác trong project |
| **NFKD / NFKC** | thêm bước thay **tương đương tương thích** | hầu như không nên dùng cho dữ liệu huấn luyện |

Khác biệt giữa "chính tắc" và "tương thích": NFKC đổi `①` thành `1`, `ﬁ` thành `fi`, dấu nháy cong thành nháy
thẳng — tức **làm mất thông tin**. Project chỉ dùng NFC và NFD.

Một chi tiết kỹ thuật của NFD cần biết vì nó ảnh hưởng tới tokenizer: khi một chữ có **hai** dấu (chất lượng
nguyên âm + thanh điệu), thứ tự hai dấu đó không tuỳ tiện. Mỗi dấu có một **lớp kết hợp chính tắc** (canonical
combining class), và NFD sắp xếp chúng tăng dần theo lớp. Dấu mũ `U+0302` có lớp 230 (trên chữ), dấu nặng
`U+0323` có lớp 220 (dưới chữ), nên trong NFD dấu nặng luôn đứng **trước** dấu mũ.

In [ ]:
dung_san = "ệ"
to_hop = "ệ"
print(f"dựng sẵn : {dung_san!r} | {len(dung_san)} code point | {[f'U+{ord(c):04X}' for c in dung_san]}")
print(f"tổ hợp   : {to_hop!r} | {len(to_hop)} code point | {[f'U+{ord(c):04X}' for c in to_hop]}")
print(f"so sánh trực tiếp bằng == : {dung_san == to_hop}")
print(f"so sánh sau khi NFC       : {nfc(dung_san) == nfc(to_hop)}")
print(f"NFD của dựng sẵn          : {[f'U+{ord(c):04X}' for c in nfd(dung_san)]}  <- dấu nặng trước dấu mũ")

print("\nlớp kết hợp chính tắc (canonical combining class):")
for c in nfd(dung_san):
    print(f"  U+{ord(c):04X} lớp {unicodedata.combining(c):3d}  {unicodedata.name(c)}")

print("\nNFD -> NFC -> NFD có quay về đúng dãy cũ không:", nfd(nfc(nfd(dung_san))) == nfd(dung_san))
print("NFKC làm mất thông tin, ví dụ:", [f"{s!r} -> {unicodedata.normalize('NFKC', s)!r}" for s in ("①", "ﬁ", "²")])

## 3. Âm tiết tiếng Việt và thanh điệu

Chữ viết tiếng Việt tổ chức theo **âm tiết**, mỗi âm tiết viết rời và gồm ba phần:

$$\text{âm tiết} = \underbrace{\text{âm đầu}}_{\text{phụ âm, có thể rỗng}} + \underbrace{\text{vần}}_{\text{nguyên âm + âm cuối}} + \underbrace{\text{thanh điệu}}_{\text{6 thanh}}$$

Sáu thanh: **ngang** (không dấu), **huyền** `◌̀`, **sắc** `◌́`, **hỏi** `◌̉`, **ngã** `◌̃`, **nặng** `◌̣`.

Phải phân biệt hai loại dấu, vì project xử lý chúng khác nhau:

| Loại | Ví dụ | Vai trò |
|---|---|---|
| **Dấu chất lượng nguyên âm** | `â` (U+0302), `ă` (U+0306), `ơ/ư` (U+031B) | đổi **nguyên âm**, là một chữ cái khác |
| **Dấu thanh** | `◌̀ ◌́ ◌̉ ◌̃ ◌̣` | đổi **thanh**, là thứ H2 và bộ cặp tối thiểu nhắm tới |

Nên "ố" ở dạng NFD là **ba** code point: `o` + dấu mũ + dấu sắc. `vitok/text.py` giữ đúng phân biệt này: khi đổi
thanh, nó chỉ thay dấu thanh và giữ nguyên dấu chất lượng.

Còn `đ` **không phải** `d` có dấu: nó là một chữ cái riêng trong bảng chữ cái tiếng Việt, và NFD **không** tách nó
ra. Vì vậy hàm bỏ dấu phải xử lý riêng:

```python
def strip_diacritics(s):
    d = "".join(ch for ch in nfd(s) if unicodedata.category(ch) != "Mn")
    return nfc(d.replace("đ", "d").replace("Đ", "D"))
```

In [ ]:
print("phân tích NFD của vài âm tiết:")
for s in ("ố", "được", "nghiêng", "đá"):
    manh = [f"{c}" if unicodedata.combining(c) == 0 else f"◌{c}({unicodedata.name(c).split()[-1].lower()})" for c in nfd(s)]
    print(f"  {s:8s} -> {len(nfd(s))} code point: {' + '.join(manh)}")

print(f"\nđ có tách được không? NFD('đ') = {[f'U+{ord(c):04X}' for c in nfd('đ')]}  -> không")
print(f"nên strip_diacritics phải thay tay: 'đá' -> {strip_diacritics('đá')!r}")

print("\nđổi thanh bằng vitok.text.with_tone (giữ nguyên dấu chất lượng nguyên âm):")
for am_tiet in ("ố", "được", "mà"):
    hien = tone_of(am_tiet)
    doi = {ten: with_tone(am_tiet, dau) for dau, ten in TONE_MARKS.items()}
    doi["ngang"] = with_tone(am_tiet, None)
    print(f"  {am_tiet!r} (thanh {TONE_MARKS.get(hien, 'ngang')}) -> {doi}")

### Một âm tiết, hai cách đặt dấu thanh

Tiếng Việt có hai quy ước đặt dấu thanh trên vần có hai nguyên âm: kiểu cũ đặt trên nguyên âm đầu ("hòa"), kiểu
mới đặt trên nguyên âm chính ("hoà"). Cả hai đều gặp trên web, và **chúng là hai dãy code point khác nhau** kể cả
sau NFC — chuẩn hoá Unicode không hợp nhất được, vì đây là khác biệt chính tả chứ không phải khác biệt mã hoá.

Hệ quả cho tokenizer: đó là hai token khác nhau, chia đôi tần suất của cùng một từ. Project **không** chuẩn hoá
việc này, và đó là một hạn chế nên ghi.

In [ ]:
kieu_cu, kieu_moi = "hòa", "hoà"
print(f"kiểu cũ  {kieu_cu!r}: {[f'U+{ord(c):04X}' for c in kieu_cu]}")
print(f"kiểu mới {kieu_moi!r}: {[f'U+{ord(c):04X}' for c in kieu_moi]}")
print(f"NFC có hợp nhất không? {nfc(kieu_cu) == nfc(kieu_moi)}")
print(f"NFD có hợp nhất không? {nfd(kieu_cu) == nfd(kieu_moi)}")

dem = Counter()
for d in docs:
    for tu in re.findall(r"\p{L}+", d):
        t = tu.lower()
        if t in ("hòa", "hoà", "thủy", "thuỷ", "khỏe", "khoẻ"):
            dem[t] += 1
print("\nsố lần gặp trong tập test:", dict(dem))

## 4. NFD trong project: nó thay đổi cái gì

Nhánh NFD của project chuẩn hoá **corpus huấn luyện tokenizer** và **văn bản đầu vào** sang NFD. Lập luận ủng hộ:
ở dạng NFD, "á", "à", "ả" chia sẻ cùng một chữ cái nền `a`, nên tokenizer có thể học `a` một lần và dùng lại cho
mọi thanh — tiết kiệm vocab, và có thể giúp model khái quát hoá quy tắc thanh điệu.

Lập luận phản đối: chuỗi dài hơn (mỗi chữ có dấu thành 2–3 code point), nên cùng một văn bản tốn nhiều token hơn,
hoặc buộc merge phải "tiêu" vào việc ghép lại chữ đã bị tách.

Số đo cho thấy phía nào thắng ở tầng tokenizer:

In [ ]:
toks = {c: Tokenizer.from_file(str(DATA / "tokenizers-16k" / c / "tokenizer.json"))
        for c in ("bpe-nfc", "bpe-nfd", "super-nfc", "super-nfd")}
thu = docs[:200]
n_char_nfc = sum(len(nfc(d)) for d in thu)

print(" điều kiện  | token   | chars NFC/token | byte/token")
for ten, t in toks.items():
    n_tok = sum(len(t.encode(d, add_special_tokens=False).ids) for d in thu)
    n_byte = sum(len((nfd(d) if "nfd" in ten else nfc(d)).encode()) for d in thu)
    print(f" {ten:10s} | {n_tok:7,} | {n_char_nfc / n_tok:15.3f} | {n_byte / n_tok:10.3f}")

print("\n=> ở tầng nén, NFC và NFD gần như bằng nhau (chênh dưới 0,1%) — nhưng byte mỗi token thì không.")


def bang_byte_unicode():
    bs = (list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1))
          + list(range(ord("®"), ord("ÿ") + 1)))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {c: b for b, c in zip(bs, (chr(x) for x in cs))}


U2B = bang_byte_unicode()


def doc_token(t):
    return bytes(U2B[c] for c in t if c in U2B).decode("utf-8", errors="replace")


def merges_cua(cond):
    raw = json.loads((DATA / "tokenizers-16k" / cond / "tokenizer.json").read_text())
    return [m if isinstance(m, str) else " ".join(m) for m in raw["model"]["merges"]]


def mo_ta(t):
    doc = doc_token(t)
    if any(unicodedata.combining(c) for c in doc):
        return f"{doc!r} <- có dấu kết hợp"
    return f"{doc!r}" if "�" not in doc else f"{doc!r} <- chưa đủ byte để thành ký tự"


print("\n12 merge đầu tiên, đặt cạnh nhau:")
print(f"  {'#':>3s} | {'bpe-nfc':<34s} | bpe-nfd")
m_nfc, m_nfd = merges_cua("bpe-nfc"), merges_cua("bpe-nfd")
for i in range(12):
    a, b = m_nfc[i].split(" ")
    c, d = m_nfd[i].split(" ")
    print(f"  {i:3d} | {mo_ta(a + b):<34s} | {mo_ta(c + d)}")


def dem_dau(merges, n=200):
    return sum(1 for m in merges[:n] if any(unicodedata.combining(c) for c in doc_token("".join(m.split(" ")))))


print(f"\ntrong 200 merge đầu: bpe-nfc có {dem_dau(m_nfc)} merge sinh ra token chứa dấu kết hợp,"
      f" bpe-nfd có {dem_dau(m_nfd)}")
print("=> nhánh NFD phải tiêu một phần ngân sách merge để ghép lại đúng thứ nó vừa tách ra.")

## 5. Cái bẫy đo lường: mẫu số phải là ký tự NFC

Đây là chỗ nhánh NFD có thể tạo ra một kết quả đẹp giả, và là lý do notebook 01 nhấn mạnh công thức

$$\mathrm{bpc} = \frac{N}{C_{\mathrm{NFC}} \ln 2}$$

Nếu chia cho số **byte** hoặc số **code point NFD**, tokenizer NFD được lợi một cách máy móc: cùng một văn bản, nó
có nhiều byte hơn 18% và nhiều code point hơn 26%, nên "bit trên đơn vị" của nó nhỏ đi mà chất lượng không đổi.

Cell dưới đo chính xác hệ số thiên lệch đó trên dữ liệu thật.

In [ ]:
c_nfc = sum(len(nfc(d)) for d in thu)
c_nfd = sum(len(nfd(d)) for d in thu)
b_nfc = sum(len(nfc(d).encode()) for d in thu)
b_nfd = sum(len(nfd(d).encode()) for d in thu)

print(" mẫu số dùng để chia | NFC        | NFD        | NFD được lợi")
print(f" code point          | {c_nfc:10,} | {c_nfd:10,} | {c_nfd / c_nfc - 1:12.1%}")
print(f" byte UTF-8          | {b_nfc:10,} | {b_nfd:10,} | {b_nfd / b_nfc - 1:12.1%}")
print("\nDùng ký tự NFC làm mẫu số cho MỌI điều kiện thì mẫu số giống hệt nhau,")
print("và mọi chênh lệch còn lại đúng là chênh lệch về chất lượng dự đoán.")

## 6. Thiết kế H2: bỏ dấu 50% và 100%

Giả thuyết H2 hỏi: tokenizer NFD có **bền hơn** khi văn bản thiếu dấu không? Đây là tình huống thật trên web tiếng
Việt — người dùng gõ không dấu rất nhiều.

Ba biến thể văn bản test, định nghĩa trong `vitok/eval.py`:

| Biến thể | Xử lý | Kiểm tra điều gì |
|---|---|---|
| `clean` | giữ nguyên | chất lượng trên văn bản chuẩn |
| `strip50` | bỏ dấu ở 50% âm tiết, chọn ngẫu nhiên với seed cố định | văn bản **lẫn lộn** có dấu và không dấu |
| `strip100` | bỏ hết dấu | văn bản không dấu hoàn toàn |

Chi tiết thiết kế đáng chú ý: `strip_diacritics` **giữ nguyên số ký tự NFC**, vì nó chỉ bỏ dấu chứ không bỏ chữ.
Nhờ vậy mẫu số của bpc không đổi khi so ba biến thể — nếu không, ba cột trong bảng kết quả sẽ không so được với
nhau.

In [ ]:
cau = "Thủ tướng Phạm Minh Chính chủ trì hội nghị trực tuyến toàn quốc về phát triển kinh tế."
print("clean   :", cau)
print("strip50 :", strip_diacritics_partial(cau, 0.5, seed=0))
print("strip100:", strip_diacritics(cau))

print(f"\nsố ký tự NFC: clean {len(nfc(cau))} | strip50 {len(nfc(strip_diacritics_partial(cau, 0.5, seed=0)))}"
      f" | strip100 {len(nfc(strip_diacritics(cau)))}   <- bằng nhau, đúng như thiết kế")

tong = {v: sum(len(nfc(f(d))) for d in thu[:50])
        for v, f in (("clean", lambda s: s), ("strip50", lambda s: strip_diacritics_partial(s, 0.5, seed=0)),
                     ("strip100", strip_diacritics))}
print("trên 50 văn bản:", tong)

print("\nsố token cần để mã hoá strip100 (văn bản không dấu):")
for ten, t in toks.items():
    a = sum(len(t.encode(d, add_special_tokens=False).ids) for d in thu[:50])
    b = sum(len(t.encode(strip_diacritics(d), add_special_tokens=False).ids) for d in thu[:50])
    print(f"  {ten:10s}: clean {a:6,} -> strip100 {b:6,} ({b / a - 1:+.1%})")

Một quan sát phụ đáng chú ý từ cell trên: bỏ dấu làm số token của **SuperBPE** phình thêm ~41%, trong khi BPE chỉ
phình ~15%. Lý do: superword là các cụm từ **có dấu** học được từ corpus ("có thể", "sử dụng"); khi văn bản mất
dấu, những token đó không khớp nữa và văn bản rơi trở lại về các mảnh nhỏ. Tức lợi thế nén của SuperBPE **không
bền** trước nhiễu chính tả — một kết luận thực dụng nên đưa vào báo cáo.

Kết quả H2 của project, đọc kèm thanh nhiễu seed từ notebook 02:

| So sánh | biến thể | d6 | d8 |
|---|---|---|---|
| bpe-nfd − bpe-nfc | strip50 | $-0{,}0115$ | $-0{,}0051$ |
| bpe-nfd − bpe-nfc | strip100 | $+0{,}0171$ | $+0{,}0007$ |
| bpe-nfd − bpe-nfc | clean (chi phí) | $+0{,}0012$ | $+0{,}0008$ |

Đọc đúng: ở d6, NFD **có lợi** trên văn bản lẫn lộn và **có hại** trên văn bản bỏ dấu hoàn toàn. Ở d8, các hiệu
ứng trên văn bản bỏ dấu ($0{,}0007$ và $0{,}0051$) không vượt chênh lệch giữa hai lần train của cùng một điều kiện
trên chính các biến thể đó ($0{,}0015$–$0{,}0095$); chi phí trên văn bản sạch ($+0{,}0008$) chỉ gấp 1,6 lần nhiễu
seed $0{,}0005$. Nên ở d8 **không kết luận được** — đúng như notebook 02 mục 11 đã lập luận. Kết luận ở d6 đứng
vững hơn: $0{,}0115$ và $0{,}0171$ đều lớn hơn nhiễu seed lớn nhất đo được ở d6 ($0{,}0068$).

## 7. Cặp tối thiểu thanh điệu

Bộ dữ liệu thứ hai của project kiểm tra một thứ khác hẳn bpc: model có **phân biệt được thanh** không. Cách làm
là lấy một câu thật, đổi thanh của **đúng một âm tiết** sang một âm tiết có thật khác, rồi hỏi model câu nào có
xác suất cao hơn.

Ràng buộc "âm tiết có thật khác" quan trọng: nếu đổi thành chuỗi không tồn tại trong tiếng Việt, model chỉ cần
nhận ra chuỗi lạ chứ không cần hiểu thanh. Vì vậy `vitok/minimal_pairs.py` chỉ đổi sang âm tiết nằm trong
`syllables.json` với tần suất tối thiểu 50.

In [ ]:
pairs = [json.loads(l) for l in (DATA / "minimal_pairs.jsonl").open(encoding="utf-8")]
print(f"{len(pairs):,} cặp tối thiểu\n")
for p in pairs[:3]:
    print(f"  đúng: {p['good']}")
    print(f"  sai : {p['bad']}")
    print(f"  đổi : {p['from']!r} -> {p['to']!r}\n")

dem_doi = Counter((tone_of(p["from"]), tone_of(p["to"])) for p in pairs)
ten = {**{k: v for k, v in TONE_MARKS.items()}, None: "ngang"}
print("10 kiểu đổi thanh hay gặp nhất:")
for (a, b), n in dem_doi.most_common(10):
    print(f"  {ten.get(a, a):6s} -> {ten.get(b, b):6s}: {n:4d} cặp")

syl = json.loads((DATA / "syllables.json").read_text())
print(f"\ntừ điển âm tiết: {len(syl):,} âm tiết, ngưỡng tần suất tối thiểu 50 giữ lại"
      f" {sum(1 for v in syl.values() if v >= 50):,}")

Độ chính xác đo được là 0,992–0,996 ở mọi điều kiện. Notebook 02 mục 9 đã chỉ ra hệ quả: chỉ còn 11–20 cặp bất
đồng giữa hai model, nên McNemar gần như không có power. Đây là **hiệu ứng chạm trần**.

Có hai nguyên nhân chồng lên nhau, và cell dưới tách chúng ra:

1. **Ít lỗi**: với độ chính xác 0,994 thì mỗi model chỉ sai khoảng 18 trong 3.000 cặp.
2. **Lỗi trùng nhau**: hai model sai ở *cùng những cặp khó*. Nếu lỗi độc lập, số cặp bất đồng kỳ vọng sẽ là
   $3000(p_A(1-p_B) + (1-p_A)p_B)$; số quan sát được nhỏ hơn hẳn, và phần chênh chính là phần tương quan.

Chỉ các cặp **bất đồng** mới mang thông tin, nên power phụ thuộc $m$ chứ không phụ thuộc 3.000.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)


def p_mcnemar(n10, n01):
    m = n10 + n01
    return 1.0 if m == 0 else min(1.0, 2 * sum(math.comb(m, i) for i in range(min(n10, n01) + 1)) / 2 ** m)


p_a, p_b = 0.995, 0.994                    # độ chính xác đo được của super-nfc và bpe-nfc ở d8
m_doc_lap = 3000 * (p_a * (1 - p_b) + (1 - p_a) * p_b)
m_quan_sat = 7 + 4                          # số thật trong results/summary.md, d8
print(f"số cặp sai của mỗi model  : {3000 * (1 - p_a):.0f} và {3000 * (1 - p_b):.0f}")
print(f"cặp bất đồng NẾU lỗi độc lập: {m_doc_lap:.0f}")
print(f"cặp bất đồng QUAN SÁT được  : {m_quan_sat}")
print(f"=> lỗi của hai model trùng nhau rất nhiều; chỉ {m_quan_sat / m_doc_lap:.0%} lượng bất đồng lý thuyết còn lại\n")

print("power của McNemar theo số cặp bất đồng m, khi A thắng 70% số cặp bất đồng:")
print("   m   | power")
for m_bd in (11, 30, 100, 300, 1000):
    thang = rng.binomial(m_bd, 0.70, size=20000)
    power = float(np.mean([p_mcnemar(int(t), m_bd - int(t)) < 0.05 for t in thang]))
    print(f"  {m_bd:4d} | {power:5.1%}")
print("\n=> cách sửa không phải tăng số cặp mà là làm cặp KHÓ hơn, để hai model bất đồng nhiều hơn.")

## 8. Âm tiết và từ: H4

Tiếng Việt viết **rời từng âm tiết**, nên khoảng trắng **không** đánh dấu ranh giới từ. "phát triển" là một từ
viết bằng hai âm tiết; "nhà tôi" là hai từ. Đây là điểm khác biệt lớn nhất so với tiếng Anh về mặt tokenizer, và
là lý do SuperBPE có thể có lợi cho tiếng Việt: token bắc qua dấu cách có cơ hội trùng với **từ thật**.

Giả thuyết H4 đo đúng điều đó. Cách đo trong `vitok/wordhood.py`:

1. Với mỗi văn bản, chạy `underthesea.word_tokenize` để lấy ranh giới từ.
2. Với mỗi **superword** (token phủ hơn một âm tiết), hỏi đoạn văn bản nó phủ có đúng là một từ hay không.
3. **Mốc so sánh**: tỷ lệ trùng của các cụm $k$ âm tiết liền nhau bất kỳ trong cùng văn bản. Nếu superword không
   quan tâm ranh giới từ, nó sẽ trùng đúng bằng mốc này.

Mốc so sánh là phần quan trọng nhất về phương pháp: không có nó, con số 65,6% không nói lên điều gì, vì trong
tiếng Việt rất nhiều cụm hai âm tiết ngẫu nhiên cũng tình cờ là từ.

In [ ]:
from underthesea import word_tokenize

for cau in ("Chính phủ đã ban hành nghị quyết về phát triển kinh tế xã hội.",
            "Nhà tôi ở gần trường học mới xây."):
    tu = word_tokenize(cau)
    print(f"  {cau}")
    print(f"    -> {tu}")
    print(f"    {sum(1 for t in tu if ' ' in t)}/{len(tu)} đơn vị là từ nhiều âm tiết\n")

sup = toks["super-nfc"]
cau = "Chính phủ đã ban hành nghị quyết về phát triển kinh tế xã hội."
manh = [sup.decode([i]) for i in sup.encode(cau, add_special_tokens=False).ids]
print("SuperBPE tách câu đó thành:")
print("   ", " | ".join(manh))
print("\nunderthesea tách thành:")
print("   ", " | ".join(word_tokenize(cau)))

In [ ]:
wh = json.loads((ROOT / "results" / "wordhood_16k.json").read_text())
v = wh["super-nfc"]
print(f"trên {wh['n_docs']} văn bản test (bỏ {v['docs_skipped']} văn bản không căn được):")
print(f"  {v['superword_occurrences']:,} lần dùng superword")
print(f"  trùng ranh giới từ thật : {v['match_rate']:.1%}")
print(f"  mốc cụm âm tiết ngẫu nhiên: {v['baseline_rate']:.1%}")
print(f"  tỷ lệ vượt mốc           : {v['match_rate'] / v['baseline_rate']:.1f}×\n")

print(" số âm tiết | số lần dùng | trùng từ thật | mốc      | vượt mốc")
for k, d in v["by_syllables"].items():
    if d["baseline_rate"]:
        print(f" {k:^10s} | {d['occurrences']:11,} | {d['match_rate']:13.1%} | {d['baseline_rate']:8.1%} |"
              f" {d['match_rate'] / d['baseline_rate']:7.1f}×")
print("\nGần như toàn bộ superword phủ đúng 2 âm tiết — cũng là độ dài phổ biến nhất của từ tiếng Việt.")
print("Với 3 âm tiết, tỷ lệ trùng tụt hẳn: SuperBPE ghép cụm hay đi cùng nhau, không phải phân tích cú pháp.")

## 9. Tóm tắt và bẫy

| Khái niệm | Điều phải nhớ |
|---|---|
| Code point vs byte | tiếng Việt NFC ≈ 1,32 byte/ký tự; NFD nhiều hơn 18% byte, 26% code point |
| NFC vs NFD | tương đương chính tắc, nhưng là hai dãy số khác nhau |
| NFKC/NFKD | làm mất thông tin, không dùng cho dữ liệu huấn luyện |
| Dấu chất lượng vs dấu thanh | `â ă ơ ư` đổi nguyên âm; `◌̀ ◌́ ◌̉ ◌̃ ◌̣` đổi thanh |
| `đ` | chữ cái riêng, NFD không tách, phải thay tay khi bỏ dấu |
| "hòa" vs "hoà" | hai chính tả hợp lệ, chuẩn hoá Unicode không hợp nhất |
| Mẫu số của bpc | luôn là ký tự **NFC**, cho mọi điều kiện |
| Âm tiết ≠ từ | khoảng trắng không đánh dấu ranh giới từ; đây là cơ hội của SuperBPE |

Bốn cái bẫy, cả bốn đều đã suýt xảy ra trong project:

1. Đếm ký tự trên chuỗi NFD → bpc của nhánh NFD đẹp giả.
2. Regex thiếu `\p{M}` → dấu NFD bị tách khỏi chữ (notebook 04 mục 3).
3. Coi `đ` là `d` có dấu → hàm bỏ dấu để sót `đ`.
4. So chuỗi bằng `==` khi hai bên khác dạng chuẩn hoá → sai ở chỗ không có thông báo lỗi.

## 10. Câu hỏi tự kiểm

1. Vì sao "ệ" ở NFD có 3 code point mà không phải 2?
2. Trong NFD, vì sao dấu nặng đứng trước dấu mũ?
3. NFKC đổi `ﬁ` thành `fi`. Vì sao điều đó không phù hợp với dữ liệu huấn luyện của project?
4. Tập test có 4.192.698 ký tự NFC. Ước lượng số code point NFD và số byte NFD của nó.
5. `strip_diacritics` giữ nguyên số ký tự NFC. Vì sao tính chất đó cần cho bảng kết quả?
6. Vì sao cặp tối thiểu chỉ đổi sang **âm tiết có thật**?
7. Nếu bỏ mốc so sánh trong H4, con số 65,6% có thể bị hiểu sai như thế nào?
8. Vì sao superword 3 âm tiết trùng từ thật kém hơn hẳn superword 2 âm tiết?

**Đáp án gợi ý**

1. Vì có hai dấu chồng: dấu mũ (chất lượng nguyên âm, cho `ê`) và dấu nặng (thanh).
2. Vì NFD sắp các dấu tăng dần theo lớp kết hợp chính tắc, mà dấu nặng (lớp 220, dưới chữ) nhỏ hơn dấu mũ
   (lớp 230, trên chữ).
3. Vì nó xoá một khác biệt có thật trong văn bản gốc; model sẽ không bao giờ thấy dạng gốc, còn khi đánh giá thì
   văn bản thật vẫn chứa dạng đó.
4. Nhân với các tỷ lệ đo ở mục 1: khoảng $4{,}19\text{M} \times 1{,}266 \approx 5{,}31$ triệu code point và
   $4{,}19\text{M} \times 1{,}32 \times 1{,}18 \approx 6{,}5$ triệu byte.
5. Vì ba cột `clean`/`strip50`/`strip100` chia cho cùng mẫu số, nên chênh lệch giữa chúng là chênh lệch về dự
   đoán chứ không phải về cách đếm.
6. Để model không thể thắng chỉ bằng cách nhận ra một chuỗi không tồn tại; bài toán phải thật sự là phân biệt thanh.
7. Có thể bị hiểu thành "SuperBPE học được từ vựng", trong khi một phần lớn tỷ lệ đó đến từ việc cụm hai âm tiết
   bất kỳ trong tiếng Việt vốn đã hay là từ (mốc 25,2%).
8. Vì tiêu chí học của SuperBPE là tần suất đi cùng nhau, không phải ranh giới từ; cụm 3 âm tiết hay gặp thường
   bắc qua ranh giới của hai từ khác nhau.

**Nguồn đọc thêm**

- [Unicode Standard Annex #15: Normalization Forms](https://unicode.org/reports/tr15/) — mục 1–3.
- [Unicode Standard Annex #29: Text Segmentation](https://unicode.org/reports/tr29/) — grapheme cluster.
- [underthesea](https://github.com/undertheseanlp/underthesea) — bộ tách từ dùng cho H4.